In [1]:
import os

In [2]:
%pwd

'd:\\Kidney_Disease_Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Kidney_Disease_Classification'

In [5]:

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list



In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [9]:
class Training:
    # Hàm khởi tạo: Nhận vào đối tượng chứa toàn bộ cấu hình huấn luyện
    def __init__(self, config: TrainingConfig):
        self.config = config

    # Hàm tải mô hình VGG16 đã tùy chỉnh (có 4 units ở lớp cuối) từ Stage 02
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    # Hàm chuẩn bị dữ liệu (Data Generator) cho tập Train và Validation
    def train_valid_generator(self):

        # Thiết lập các tham số chung cho Data Generator:
        # - rescale: Chuẩn hóa giá trị điểm ảnh từ [0, 255] về [0, 1]
        # - validation_split: Trích 20% dữ liệu làm tập Validation kiểm thử
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        # Thiết lập cấu hình đọc ảnh:
        # - target_size: Đưa kích thước ảnh về (224, 224) bỏ qua số kênh màu (3)
        # - batch_size: Số lượng ảnh nạp vào mô hình trong mỗi lần tính toán
        # - interpolation: Thuật toán nội suy song tuyến tính để resize ảnh mịn hơn
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        # Tạo Bộ phát dữ liệu (Generator) cho tập Validation
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        # >>> NƠI NHẬN DIỆN 4 UNITS (TẬP VALIDATION) <<<
        # flow_from_directory sẽ tự quét thư mục self.config.training_data:
        # 1. Phát hiện 4 thư mục con (Cyst, Normal, Stone, Tumor).
        # 2. Xếp theo thứ tự ABC -> Tạo vector One-Hot 4 phần tử tương ứng 4 units.
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation", # Lấy 20% dữ liệu đã chia ở trên
            shuffle=False,       # Không xáo trộn tập validation để dễ đánh giá
            **dataflow_kwargs
        )

        # Kiểm tra nếu cấu hình cho phép Tăng cường dữ liệu (Augmentation)
        if self.config.params_is_augmentation:
            # Tạo dữ liệu giả lập giúp mô hình tránh học vẹt (Overfitting)
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,      # Xoay ảnh ngẫu nhiên tối đa 40 độ
                horizontal_flip=True,   # Lật ngang ảnh ngẫu nhiên
                width_shift_range=0.2,  # Dịch chuyển ảnh theo chiều ngang 20%
                height_shift_range=0.2, # Dịch chuyển ảnh theo chiều dọc 20%
                shear_range=0.2,        # Cắt nghiêng ảnh 20%
                zoom_range=0.2,         # Thu/phóng ảnh ngẫu nhiên 20%
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        # >>> NƠI NHẬN DIỆN 4 UNITS (TẬP TRAINING) <<<
        # Tiếp tục đọc 4 thư mục bệnh và gán nhãn chính thức cho tập Train (80% còn lại)
        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",   # Lấy 80% dữ liệu dùng để học
            shuffle=True,        # Xáo trộn ảnh để mô hình học khách quan
            **dataflow_kwargs
        )

    # Hàm hỗ trợ lưu mô hình ra file .h5
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    # Hàm thực thi quá trình huấn luyện chính
    def train(self):
        # Tính số bước (steps) trong 1 Epoch cho tập Train = Tổng số ảnh Train / Batch Size
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        # Tính số bước trong 1 Epoch cho tập Valid = Tổng số ảnh Valid / Batch Size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        # Kích hoạt quá trình huấn luyện mô hình (Fit)
        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        # Lưu mô hình đã train xong vào artifacts/training/model.h5
        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [13]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e



[2026-09-12 15:40:00,198: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-12 15:40:00,202: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-12 15:40:00,204: INFO: common: created directory at: artifacts]
[2026-09-12 15:40:00,205: INFO: common: created directory at: artifacts\training]
Found 2487 images belonging to 4 classes.
Found 9959 images belonging to 4 classes.
311/311 [==============================] - 882s 3s/step - loss: 16.1988 - accuracy: 0.3696 - val_loss: 5.3841 - val_accuracy: 0.6319
